# Preprocessing and Feature Extraction

## Overview and Approach
This notebook is dedicated to transforming raw, continuous EEG recordings into a structured mathematical format suitable for Machine Learning classification (Alzheimer's vs. Frontotemporal Dementia vs. Healthy Controls).

While the dataset (OpenNeuro ds004504) has already been robustly cleaned of artifacts by the original authors using ASR and ICA, raw time-series data is highly dimensional, non-stationary, and poorly . 

To suit the data for standard machine learning classifiers this, our strategy relies on **Feature Engineering via Spectral Analysis**. Instead of feeding raw signal amplitudes into our models, we will translate the data from the time domain to the frequency domain to extract neurophysiological biomarkers.

### Pipeline
* **Epoching (Data Windowing):** We divide the continuous EEG into 4-second windows with a 50% overlap. This serves two purposes: it ensures the signal is relatively stable (stationary) within each window, and it artificially increases our sample size, providing the ML models with thousands of short, labeled examples rather than just 88 long ones.
* **Power Spectral Density (PSD):** We use Welch's method to calculate the PSD. This acts as a mathematical equalizer, showing us the power of the brain's electrical activity at different frequencies.
* **Relative Band Power (RBP):** We extract the relative energy of five specific brainwave bands (Delta, Theta, Alpha, Beta, and Gamma). The shift of brain power from faster bands (Alpha/Beta) to slower bands (Delta/Theta) is a strong clinical indicator of neurodegeneration. 
* **Export:** By the end of this notebook, we will have converted massive continuous EEG files into a clean, tabular dataset of localized brainwave features, ready for training robust classification models.

## 1. Imports and Environment Setup

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import mne
from scipy.signal import welch, hilbert



# Suppress MNE verbosity for cleaner notebook output
mne.set_log_level('WARNING')

# Define paths (make some changes here to match your directory structure)
DATA_DIR = '../data/raw/ds004504/derivatives/'
PROCESSED_DIR = '../data/processed/'

# Create processed directory if it doesn't exist just in case
os.makedirs(PROCESSED_DIR, exist_ok=True)

### A Quick View of the Data

In [ ]:

# 1. Define paths and select a specific subject
subject_id = 'sub-001'

# Find the .set file for the selected subject
set_file_path = glob.glob(os.path.join(DATA_DIR, subject_id, 'eeg', '*.set'))

if not set_file_path:
    print(f"File not found for {subject_id}. Please check the directory structure.")
else:
    set_file = set_file_path[0]
    
    # Load the continuous preprocessed EEG data
    raw = mne.io.read_raw_eeglab(set_file, preload=True)
    
    # 2. Plot a segment of the continuous data
    # We display only 5 channels and 10 seconds to keep the plot easier to view
    print(f"Plotting data for {subject_id}...")
    raw.plot(duration=10, n_channels=5, 
                       title="Continuous EEG - First 10 Seconds")

## 2. Let's Define Frequency Bands and Extraction Function

We need to calculate the relative ratio of PSD for each epoch. The bands are defined as:
Here is the annotated list with brief English comments explaining what each band represents and its clinical relevance for our dementia (AD/FTD) classification analysis:

* **Delta:** 0.5 - 4 Hz — Represents deep sleep and restorative states.
    * **Useful:** Pathological increases in delta power (abnormal slowing of brain waves) are frequently observed in advanced stages of Alzheimer's Disease (AD) and dementia.
* **Theta:** 4 - 8 Hz — Represents drowsiness, meditation, and light sleep.
    * **Highly Useful:** A significant increase in theta power is one of the most prominent and consistent biomarkers for cognitive decline in both AD and FTD.
* **Alpha:** 8 - 13 Hz — Represents relaxed wakefulness, particularly when the eyes are closed.
    * **Crucial:** Because our dataset is specifically "resting-state, eyes-closed", measuring the reduction in alpha power is arguably the most important feature. Healthy controls should have strong alpha peaks, while AD patients typically show weakened alpha activity. 
* **Beta:** 13 - 25 Hz — Represents active thinking, focus, alertness, and problem-solving.
    * **Useful:** Usually shows decreased power in dementia patients compared to healthy controls, reflecting an overall decline in active cognitive processing.
* **Gamma:** 25 - 45 Hz — Represents high-level cognitive processing, perception, and memory consolidation.
    * **Useful:** Helps the machine learning models identify breakdowns in complex neural network connectivity, which are common in neurodegenerative conditions.


> **Note on PSD (Power Spectral Density):** Think of PSD as an audio equalizer for brain waves. It transforms the chaotic, tangled EEG signal from the *time domain* into the *frequency domain*. In simpler terms, it tells us exactly how much "power" (energy) the brain is emitting at each specific frequency. This allows us to quantify how much of the brain's activity is Delta, Theta, Alpha, Beta, or Gamma.

> **Note on RBM (The Relative Band Power)**: RBP is the PSD of the specific band divided by the total PSD of the 0.5–45 Hz range


### Advanced Feature Engineering: Functional Connectivity

In neurodegenerative disorders, the breakdown of communication between brain regions is often as informative as local spectral power. We use **Phase Locking Value (PLV)** , which quantifies the consistency of phase differences between two signals and is robust to amplitude fluctuations.

Because our dataset already enforces subject‑wise separation, we can safely compute PLV **within each epoch** without leaking subject identity.

To avoid excessive computational cost and overfitting, we restrict the analysis to **10 clinically relevant channel pairs** (fronto‑temporal and fronto‑parietal connections). This yields only **10 additional features per epoch**, keeping the total feature count manageable.

> ⚠️ **Dimensionality Note**  
> Full pairwise connectivity (171 pairs) would add 171–342 features. If needed, this can be enabled by modifying the `calculate_connectivity_fast` function, but it would require stronger feature selection in Notebook 02.

We will now define the optimized connectivity extraction function and integrate it into the processing loop.

In [ ]:
BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Beta': (13, 25),
    'Gamma': (25, 45)
}

def calculate_rbp(epoch_data, sfreq, ch_names):
    """
    Calculates the Relative Band Power for a single epoch.
    epoch_data: array of shape (n_channels, n_times)
    sfreq: sampling frequency (500 Hz for this dataset)
    ch_names: list of channel names from raw.info['ch_names']
    """
    n_channels = epoch_data.shape[0]
    features = {}
    
    for ch_idx in range(n_channels):
        # Calculate PSD using Welch's method
        freqs, psd = welch(epoch_data[ch_idx], fs=sfreq, nperseg=sfreq*2)
        
        # Calculate total power (0.5 to 45 Hz)
        total_idx = np.logical_and(freqs >= 0.5, freqs <= 45)
        total_power = np.trapezoid(psd[total_idx], freqs[total_idx])
        
        if total_power == 0:
            total_power = 1e-10
            
        # Calculate relative power for each band
        for band_name, (fmin, fmax) in BANDS.items():
            band_idx = np.logical_and(freqs >= fmin, freqs <= fmax)
            band_power = np.trapezoid(psd[band_idx], freqs[band_idx])
            
            rbp = band_power / total_power
            feature_name = f'{ch_names[ch_idx]}_{band_name}'
            features[feature_name] = rbp
            
    return features


# ============================================================
# Functional Connectivity Features
# ============================================================

def calculate_connectivity_fast(epoch_data, sfreq, ch_names, pairs=None):
    """
    Fast connectivity: only PLV for selected channel pairs.
    """
    if pairs is None:
        pairs = [
            ('Fp1', 'F3'), ('Fp2', 'F4'), ('F3', 'C3'), ('F4', 'C4'),
            ('T3', 'T5'), ('T4', 'T6'), ('P3', 'O1'), ('P4', 'O2'),
            ('Fz', 'Cz'), ('Cz', 'Pz')
        ]
    
    features = {}
    ch_idx_map = {name: idx for idx, name in enumerate(ch_names)}
    
    for ch1_name, ch2_name in pairs:
        if ch1_name not in ch_idx_map or ch2_name not in ch_idx_map:
            continue
        i = ch_idx_map[ch1_name]
        j = ch_idx_map[ch2_name]
        
        phase1 = np.angle(hilbert(epoch_data[i]))
        phase2 = np.angle(hilbert(epoch_data[j]))
        plv = np.abs(np.mean(np.exp(1j * (phase1 - phase2))))
        features[f'plv_{ch1_name}-{ch2_name}'] = plv
    
    return features

## 3. Process Subjects and Build Feature Matrix

We will iterate through all subject folders, load their `.set` file, apply the 4-second epoching with 50% overlap, extract the features, and append them to a dataset alongside their diagnostic label.

**Why 4-second epochs with a 50% overlap?**

* **4-second duration:** In EEG signal processing, a 4-second window provides an optimal balance for calculating the Power Spectral Density (PSD). It is long enough to capture low-frequency waves (like Delta) and provides excellent frequency resolution.

* **50% overlap:** When calculating the PSD using Welch's method, a mathematical filter (a "window function") is applied to each epoch. This function tapers the edges of the signal down to zero to prevent artificial noise where the cut was made. However, this means any brain activity happening at the edges of the epoch is attenuated (squashed) and essentially lost. By overlapping the epochs by 50%, the attenuated edges of one epoch become the preserved center of the next, ensuring no critical neural information is missed by the machine learning model.
    * If we didn't perform the overlay, all brain activity occurring near the "edges" (at second 0, second 4, second 8) would be attenuated and almost lost in the analysis.

### A Quick View of the Epoched Data

Let's visualize how the epoching process transforms it. 
We will slice the continuous data of `sub-001` into **4-second windows** with a **50% overlap** (a 2-second step). 

Plotting the epochs allows us to verify that the mathematical slicing aligns correctly before we extract the Power Spectral Density (PSD) features.

In [ ]:
# Find the .set file for the selected subject
set_file_path = glob.glob(os.path.join(DATA_DIR, subject_id, 'eeg', '*.set'))

if not set_file_path:
    print(f"File not found for {subject_id}. Please check the directory structure.")
else:
    set_file = set_file_path[0]
    
    # Load the continuous preprocessed EEG data
    raw = mne.io.read_raw_eeglab(set_file, preload=True)
    
    # 2. Apply the epoching logic
    # Duration: 4 seconds | Overlap: 2 seconds (50%)
    epochs = mne.make_fixed_length_epochs(raw, duration=4.0, overlap=2.0, preload=True)
    
    print(f"Total epochs created for {subject_id}: {len(epochs)}")
    print(f"Plotting epoched data for {subject_id}...")
    
    # 3. Plot the epoched data
    # We display the first 3 epochs and 5 channels to keep the visualization clean
    fig_epochs = epochs.plot(n_epochs=3, n_channels=5, 
                             title="Epoched EEG - First 3 Windows (4s duration, 50% overlap)")

In [ ]:
# Load participants metadata (for labels)
participants_df = pd.read_csv('../data/raw/ds004504/participants.tsv', sep='\t')

# Create a dictionary for quick label lookup { 'sub-001': 'A', 'sub-066': 'C', etc. }
label_map = dict(zip(participants_df['participant_id'], participants_df['Group']))

all_features = []

# Find all subject folders in the derivatives directory
subject_folders = sorted([d for d in os.listdir(DATA_DIR) if d.startswith('sub-')])

print(f"Found {len(subject_folders)} subjects. Starting processing...")

for sub in subject_folders:
    # The dataset metadata uses 'A' for AD, 'F' for FTD, and 'C' for CN
    group_label = label_map.get(sub, 'Unknown')

    # Path to the .set file
    set_file_path = glob.glob(os.path.join(DATA_DIR, sub, 'eeg', '*.set'))

    if not set_file_path:
        print(f"Missing .set file for {sub}")
        continue

    set_file = set_file_path[0]

    try:
        # Load preprocessed EEG data
        raw = mne.io.read_raw_eeglab(set_file, preload=True)

        # Defensive: keep only EEG channels and exclude any channels marked as bad.
        # For ds004504 this is likely a no-op since the EEGLAB pipeline already
        # outputs clean EEG-only files, but it prevents silent errors if any
        # non-EEG channel (e.g. status, EOG) survived preprocessing.
        raw.pick_types(eeg=True, exclude='bads')

        # Extract sampling frequency and channel names once per subject
        sfreq = raw.info['sfreq']
        ch_names = raw.info['ch_names']

        # Epoch into 4-second windows with 50% overlap (2-second step)
        epochs = mne.make_fixed_length_epochs(raw, duration=4.0, overlap=2.0, preload=True)
        epochs_data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)

        # Extract RBP + Connectivity features for each epoch
        for i, epoch in enumerate(epochs_data):
            # 1. Spectral band power features
            epoch_features = calculate_rbp(epoch, sfreq, ch_names)
            
            # 2. Functional connectivity features 
            conn_features = calculate_connectivity_fast(epoch, sfreq, ch_names)
            epoch_features.update(conn_features)   # Merge dictionaries
            
            # 3. Metadata
            epoch_features['subject_id'] = sub
            epoch_features['epoch_id'] = i
            epoch_features['label'] = group_label
            
            all_features.append(epoch_features)

    except Exception as e:
        print(f"Error processing {sub}: {e}")

print("Processing complete!")

## 4. Save Final Dataset

Convert the list of dictionaries into a Pandas DataFrame, map the channel indices back to their standard 10-20 names if necessary, and export to CSV.

### Final Feature Set Size
- 19 channels × 5 bands = **95** RBP features
- 10 clinically relevant channel pairs × PLV = **10** connectivity features
- **Total features per epoch = 105**

In [ ]:
# Build DataFrame
features_df = pd.DataFrame(all_features)

# Reorder columns to have metadata first
cols = ['subject_id', 'epoch_id', 'label'] + [c for c in features_df.columns if c not in ['subject_id', 'epoch_id', 'label']]
features_df = features_df[cols]

# Map labels for clarity: 'A' -> 'AD', 'F' -> 'FTD', 'C' -> 'CN'
features_df['label'] = features_df['label'].map({'A': 'AD', 'F': 'FTD', 'C': 'CN'})

# Drop any rows with unknown labels
features_df = features_df.dropna(subset=['label'])

display(features_df.head())

# --- Validation (before saving) ---
print(f"Shape: {features_df.shape}")
print(features_df['label'].value_counts())
print(f"Unique subjects: {features_df['subject_id'].nunique()}")
print(f"NaN values: {features_df.isnull().sum().sum()}")
print(f"Inf values: {np.isinf(features_df.select_dtypes('number').values).sum()}")

# Abort if data is corrupted
assert features_df.isnull().sum().sum() == 0, "NaN values found — check feature extraction."
assert not np.isinf(features_df.select_dtypes('number').values).any(), "Inf values found — check band ratio eps."

# --- Save ---
output_path = os.path.join(PROCESSED_DIR, 'eeg_rbp_features.csv')
features_df.to_csv(output_path, index=False)
print(f"Feature matrix saved to {output_path} with shape {features_df.shape}")

> **Next Step:** With the feature extraction complete, you can now proceed to the [model training notebook](02_models_training.ipynb).